# Lesson 9 — Ship your mini-GPT

Last step: hide everything behind one clean interface. `MiniGPT` loads the trained weights and has one method,
`complete(prompt)`. That's the same shape as calling any LLM API: prompt in, text out.

Part 1 is the course code in the notebook. Part 2 imports the same thing from my module `mini_gpt.py` to show it
works as a reusable file. (The portfolio project is separate and not part of this notebook.)

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(1337)

## Part 1 — the course version
### Vocabulary + config (must match what `gpt.pt` was trained with)

In [2]:
text = open("shakespeare.txt").read()
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]

# --- model config (small, so it trains on a CPU in a couple of minutes) ---
block_size = 32     # context length
n_embd     = 64     # embedding size
n_heads    = 4      # attention heads
n_layers   = 3      # transformer blocks
batch_size = 16

### Model classes (same as lessons 7–8)

In [3]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj  = nn.Linear(n_heads * head_size, n_embd)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        self.sa  = MultiHeadAttention(n_heads, n_embd // n_heads)
        self.ff  = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd); self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, n_heads) for _ in range(n_layers)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T))
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)                       # (B, T, vocab)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

### The wrapper
`MiniGPT` loads the weights once. `complete()` encodes the prompt, then samples with temperature 0.8 and top-k 20
(a sensible middle setting from lesson 8) and decodes back to text.

In [4]:
class MiniGPT:
    "Load a trained GPT and generate text from a prompt."
    def __init__(self, weights="gpt.pt"):
        self.model = GPT(); self.model.load_state_dict(torch.load(weights)); self.model.eval()

    @torch.no_grad()
    def complete(self, prompt="\n", max_new_tokens=300, temperature=0.8, top_k=20):
        idx = torch.tensor([encode(prompt)], dtype=torch.long)
        for _ in range(max_new_tokens):
            logits, _ = self.model(idx[:, -block_size:])
            logits = logits[:, -1, :] / temperature
            kth = torch.topk(logits, top_k)[0][:, [-1]]
            logits = logits.masked_fill(logits < kth, float("-inf"))
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return decode(idx[0].tolist())

In [5]:
gpt = MiniGPT()
romeo_nb = gpt.complete(prompt="ROMEO:", max_new_tokens=200)
print(romeo_nb)

ROMEO:
ENot whir homare:
My for sids not you,
Se:
Oy nom wald mong ande dove she ill,
thig for a
Oid my lighs geere tis a the thit but wightood whice,
Nond frer of of gor his hout hat fould here,
Wher all'd


## Reference check against the course output

In [6]:
course_romeo = "ROMEO:\nEncthe If hord and, the my asent ing spon,\nMy nove ald mongor te dove she ill,"
same = next((i for i, (a, b) in enumerate(zip(romeo_nb, course_romeo)) if a != b), len(course_romeo))
print("starts with the prompt 'ROMEO:'   :", romeo_nb.startswith("ROMEO:"))
print("length = 6 prompt + 200 new chars :", len(romeo_nb) == 6 + 200, f"({len(romeo_nb)})")
print("shared prefix with course sample  :", same, "chars")
print("   course:", repr(course_romeo[:60]))
print("   mine  :", repr(romeo_nb[:60]))

starts with the prompt 'ROMEO:'   : True
length = 6 prompt + 200 new chars : True (206)
shared prefix with course sample  : 8 chars
   course: 'ROMEO:\nEncthe If hord and, the my asent ing spon,\nMy nove al'
   mine  : 'ROMEO:\nENot whir homare:\nMy for sids not you,\nSe:\nOy nom wal'


The interface behaves like the course's: the prompt `ROMEO:` is kept, exactly 200 characters are added, and
the result is short lines with capitalised starts and `word:` tags. The text itself differs after `ROMEO:\nE`
(8 characters) because my `gpt.pt` from lesson 7 differs from the course's by tiny rounding amounts (loss within 0.002).

| course | mine |
|---|---|
| `My nove ald mongor te dove she ill,` | `Oy nom wald mong ande dove she ill,` |
| `Wher are'd` (last line) | `Wher all'd` (last line) |

As in lesson 8, the two runs keep re-syncing on the same fragments (`dove she ill,`, `Wher a…'d`), because both use
seed 1337 and the models are nearly the same.

## Part 2 — the same thing as a module: `mini_gpt.py`
I put the model classes and `MiniGPT` into `work/mini_gpt.py`. It finds `gpt.pt` and `shakespeare.txt` in its own
folder, so it works no matter where it's imported from. Here I import it fresh and check that it really is
the same model as above.

In [7]:
import importlib, mini_gpt
importlib.reload(mini_gpt)
print("module file      :", mini_gpt.__file__.split("/")[-1])
print("weights it loads :", mini_gpt.MiniGPT.__init__.__defaults__[0].name)

shipped = mini_gpt.MiniGPT()
print("parameters       :", sum(p.numel() for p in shipped.model.parameters()))
same_weights = all(torch.equal(a, b) for a, b in zip(shipped.model.state_dict().values(), gpt.model.state_dict().values()))
print("identical weights to Part 1:", same_weights)

module file      : mini_gpt.py
weights it loads : gpt.pt
parameters       : 159937
identical weights to Part 1: True


In [8]:
torch.manual_seed(1337); a = gpt.complete(prompt="ROMEO:", max_new_tokens=120)
torch.manual_seed(1337); b = shipped.complete(prompt="ROMEO:", max_new_tokens=120)
print("same seed -> same text from notebook class and module:", a == b)
print()
torch.manual_seed(7)
print(shipped.complete(prompt="JULIET:\n", max_new_tokens=200))

same seed -> same text from notebook class and module: True

JULIET:
Whou, my bement oure leftoe you seater to hit be myest my hat me borde whate shats is a cherpe berse, thave ens heret; he old hit the a
Whoure and gonce lout toun weson thasive;
Ged plass lam my fasee


## Test yourself

**1. Why does `MiniGPT` need the exact same config (block_size, n_embd, …) as training?**
`load_state_dict` copies tensors by name and shape. A different `n_embd` or layer count builds a model whose
tensors don't fit, and loading fails.

**2. Why does `mini_gpt.py` look for `gpt.pt` next to itself instead of in the current folder?**
So it works no matter where Python is started from. A relative `"gpt.pt"` is resolved against the working
directory, which changes depending on who imports the module.

**3. How is `complete()` like calling a chat API?**
Prompt in, text out, with sampling knobs (temperature, top-k). The API hides a much bigger model but the loop is
the same: predict next token, sample, append, repeat.

## My notes
- Wrapping the model in one object with one method makes my GPT feel like a product — the same interface as a real LLM.
- Surprising: the whole "shippable" model is a 0.7 MB weights file plus ~100 lines of code.